# TrackViewer Visualization of Filtered STRs and Functional Annotations

This notebook uses the [trackViewer](https://bioconductor.org/packages/trackViewer) (Bioconductor) R package to visualize the **8 filtered STRs** from `Relatorio_STR_Final_Integral.pdf` together with their functional annotations.

**Inputs**
- Local project data: STR coordinates / residual / scRNA-seq expression
- External tracks downloaded by `7.4.3.1_download_external_tracks.sh` into `external_tracks/`

**Reference genome:** hg38 (GRCh38).

## 1. Environment setup

Install packages (run once):
```r
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("trackViewer", "GenomicRanges", "rtracklayer", "TxDb.Hsapiens.UCSC.hg38.knownGene"))
install.packages("readr")
```

In [ ]:
suppressPackageStartupMessages({
  library(trackViewer)
  library(GenomicRanges)
  library(rtracklayer)
  library(TxDb.Hsapiens.UCSC.hg38.knownGene)
  library(readr)
})

cat('trackViewer loaded OK\n')

## 2. Define the 8 filtered STRs

Coordinates are 0-based start (BED-style) matching the report and `STR_variants_UCSC_track.bed`.

In [ ]:
# (chr, start0, end, STRs_ID, gene, abs_res, allele2, group, motif, motif_size)
variants <- read.delim(textConnection('chr\tstart0\tend\tSTRs_ID\tgene\tabs_res\tallele2\tgroup\tmotif\tcopy\n'
  "chr1\t211045040\t211045050\tchr1:211045041:AT:9\tKCNH1\t30.13\t31.20\tControl\tAT\t9\n"
  "chr1\t76143391\t76143408\tchr1:76143392:GT:16\tST6GALNAC3\t13.38\t24.90\tControl\tGT\t16\n"
  "chr3\t76185194\t76185206\tchr3:76185195:AT:11\tROBO2\t26.15\t18.47\tControl\tAT\t11\n"
  "chr5\t22138844\t22138861\tchr5:22138845:AT:16\tCDH12\t61.07\t22.76\tControl\tAT\t16\n"
  "chr6\t73097141\t73097151\tchr6:73097142:AT:9\tKCNQ5\t23.64\t14.85\tControl\tAT\t9\n"
  "chr6\t123976247\t123976256\tchr6:123976248:AT:8\tNKAIN2\t43.71\t21.22\tControl\tAT\t8\n"
  "chr10\t60288888\t60288908\tchr10:60288889:AC:19\tANK3\t20.60\t16.75\tControl\tAC\t19\n"
  "chr15\t47551672\t47551687\tchr15:47551673:GT:14\tSEMA6D\t33.20\t14.42\tCase\tGT\t14\n'),
  header = TRUE, sep = '\t', stringsAsFactors = FALSE)

print(variants[, c('STRs_ID','gene','abs_res','allele2','group')])

# GRanges of the STRs (1-based start)
gr_strs <- GRanges(seqnames = variants$chr,
                   ranges = IRanges(start = variants$start0 + 1, end = variants$end),
                   strand = '*',
                   STRs_ID = variants$STRs_ID,
                   gene = variants$gene,
                   abs_res = variants$abs_res,
                   allele2 = variants$allele2,
                   group = variants$group,
                   motif = variants$motif)
names(gr_strs) <- variants$gene
gr_strs

## 3. Load external tracks

Files are expected in `external_tracks/` (created by the download script). Each track is a helper function that returns a `trackViewer` feature track (importing bigWig/BED via `rtracklayer::import`).

In [ ]:
ext_dir <- 'external_tracks'
stopifnot(dir.exists(ext_dir))

# Track building helpers -----------------------------------------------------
import_bw <- function(path, name, color) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  gr <- import(path)
  if (length(gr) == 0) return(NULL)
  tr <- trackViewer::new("Track", dat = gr, type = "data", name = name)
  tr@color <- color
  tr
}

import_bed <- function(path, name, color) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  gr <- import(path, format = 'BED')
  if (length(gr) == 0) return(NULL)
  tr <- trackViewer::new("Track", dat = gr, type = "feature", name = name)
  tr@color <- color
  tr
}

# Individual annotation files -----------------------------------------------
tracks_external <- list(
  CTCF   = import_bw(file.path(ext_dir, 'CTCF_ENCFF341CQE.bw'),       'CTCF ChIP-seq',    '#D7301F'),
  DNase  = import_bw(file.path(ext_dir, 'DNase_brain.bw'),            'DNase',            '#E08214'),
  H3K27ac= import_bw(file.path(ext_dir, 'H3K27ac_brain.bw'),          'H3K27ac',          '#8073AC'),
  RemapD = import_bw(file.path(ext_dir, 'remap2022_density_hg38.bw'), 'ReMap Density',    '#4575B4')
)

# ReMap TF peak tracks (any available)
tf_names <- c('NACC2','FOXB1','KLF9','GATA2','ZNF384','MNT')
tf_files <- c(
  'remap2022_nacc2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_foxb1_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_klf9_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_gata2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_znf384_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_mnt_all_macs2_hg38_v1_0.bed.gz'
)
for (i in seq_along(tf_names)) {
  tr <- import_bed(file.path(ext_dir, tf_files[i]), tf_names[i], '#238B45')
  if (!is.null(tr)) tracks_external[[tf_names[i]]] <- tr
}

cat('External tracks loaded:', sum(!vapply(tracks_external, is.null, logical(1))), '/', length(tracks_external), '\n')

## 4. Load local project data (scRNA-seq expression)

Optional overlay: LogFC per cell type for the STR genes from the unified scRNA-seq overlap file.

In [ ]:
scrna_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
if (file.exists(scrna_file)) {
  scrna <- read_csv(scrna_file, show_col_types = FALSE)
  scrna <- scrna[!is.na(scrna$LogFC) & !is.na(scrna$STRs_ID), ]
  cat('scRNA overlap rows:', nrow(scrna), '\n')
  print(unique(scrna[, c('gene_name','STRs_ID','source_tissue','LogFC')]))
} else {
  cat('scRNA overlap file not found; skipping overlay\n')
  scrna <- NULL
}

## 5. Gene model track

Build a gene track per locus from the TxDb (hg38 knownGene).

In [ ]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

gene_track_for <- function(gr_variant, genes_of_interest) {
  # extend window a bit to capture the gene context
  win <- gr_variant
  start(win) <- start(win) - 20000
  end(win)   <- end(win)   + 20000
  geneModelFromTxdb(txdb, gr = win,
                    transcriptIds = NULL, geneSymbols = genes_of_interest)
}

cat('Gene model helper ready\n')

## 6. Variant track with annotations

Each STR is drawn as a feature with height scaled by absolute residual and colored by group.

In [ ]:
make_str_track <- function(gr_variant) {
  gr <- gr_variant
  col <- ifelse(gr$group == 'Case', '#C62828', '#1565C0')
  tr <- trackViewer::new("Track", dat = gr, type = "variant", name = gr$gene)
  tr@color <- col
  # label with residual
  tr@height <- 0.4
  tr
}

cat('Variant track helper ready\n')

## 7. scRNA-seq expression overlay (optional track)

Creates a data track with LogFC values per STR locus (mean across cell types).

In [ ]:
make_scrna_track <- function(scrna, gr_variant) {
  if (is.null(scrna)) return(NULL)
  sub <- scrna[scrna$STRs_ID == gr_variant$STRs_ID, ]
  if (nrow(sub) == 0) return(NULL)
  gr <- gr_variant
  mcols(gr)$score <- mean(sub$LogFC, na.rm = TRUE)
  tr <- trackViewer::new("Track", dat = gr, type = "data", name = paste0(gr_variant$gene, ' LogFC'))
  tr@color <- '#6A51A3'
  tr
}

cat('scRNA overlay helper ready\n')

## 8. Render one figure per STR

For each of the 8 loci, `viewTracks` combines: gene model, external functional tracks, scRNA LogFC and the STR variant. Figures are saved as PNG (and optionally PDF).

In [ ]:
dir.create('results', showWarnings = FALSE)

render_variant <- function(gr_variant, track_list, out_png) {
  viewTracks(track_list, gr = gr_variant,
             viewerStyle = trackViewerStyle(theme = 'college'),
             autoOptimizeStyle = FALSE)
  # save current device to PNG
  dev.print(png, file = out_png, width = 1400, height = 900, res = 150)
  cat('saved:', out_png, '\n')
}

track_list_names <- names(tracks_external)

for (i in seq_len(nrow(variants))) {
  gr_v <- gr_strs[i]
  
  trackList <- list()
  
  # gene model
  gt <- tryCatch(gene_track_for(gr_v, gr_v$gene), error = function(e) NULL)
  if (!is.null(gt)) trackList[[1]] <- gt
  
  # external functional tracks
  for (nm in track_list_names) {
    tr <- tracks_external[[nm]]
    if (!is.null(tr)) {
      # subset track to a window around the variant (faster, cleaner)
      win <- resize(gr_v, width = 40000, fix = 'center')
      tr_sub <- trackViewer::new('Track', dat = subsetByOverlaps(tr@dat, win), type = tr@type, name = tr@name)
      tr_sub@color <- tr@color
      trackList[[length(trackList) + 1]] <- tr_sub
    }
  }
  
  # scRNA LogFC overlay
  sct <- make_scrna_track(scrna, gr_v)
  if (!is.null(sct)) trackList[[length(trackList) + 1]] <- sct
  
  # STR variant
  trackList[[length(trackList) + 1]] <- make_str_track(gr_v)
  
  out_png <- sprintf('results/trackviewer_%s.png', gr_v$gene)
  render_variant(gr_v, trackList, out_png)
}

cat('\nAll variant figures generated under results/\n')

## 9. Combined panel (optional)

All 8 loci can be combined into a single `browseTracks` interactive page.

In [ ]:
if (interactive()) {
  # collect tracks for all variants and open interactive browser
  browseTracks(trackList)  # last trackList of the loop
}
cat('Done.\n')